In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
sentences = [
    "i love ai",
    "i love python",
    "i hate bugs",
    "bugs are annoying",
    "python is amazing",
    "i hate errors"
]

labels = [1, 1, 0, 0, 1, 0]

In [3]:
vocab = {"<PAD>": 0, "<UNK>": 1}

for sentence in sentences:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = len(vocab)

print(vocab)

{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'ai': 4, 'python': 5, 'hate': 6, 'bugs': 7, 'are': 8, 'annoying': 9, 'is': 10, 'amazing': 11, 'errors': 12}


In [4]:
def encode(sentence):
    return [vocab.get(word, vocab["<UNK>"]) for word in sentence.split()]

encoded_sentences = [encode(sentence) for sentence in sentences]

print(encoded_sentences)

[[2, 3, 4], [2, 3, 5], [2, 6, 7], [7, 8, 9], [5, 10, 11], [2, 6, 12]]


In [5]:
max_len = max(len(sentence) for sentence in encoded_sentences)

padded_sentences = [
    sentence + [vocab["<PAD>"]] * (max_len - len(sentence))
    for sentence in encoded_sentences
]

print(padded_sentences)

[[2, 3, 4], [2, 3, 5], [2, 6, 7], [7, 8, 9], [5, 10, 11], [2, 6, 12]]


In [6]:
X = torch.tensor(padded_sentences)
y = torch.tensor(labels)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [7]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 12)
        self.lstm = nn.LSTM(12, 20, batch_first=True)
        self.fc = nn.Linear(20, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        output = self.fc(hidden[-1])
        return output

model = SentimentLSTM(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [8]:
for epoch in range(100):

    model.train()

    correct = 0
    total = 0

    for inputs, targets in loader:

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    if (epoch + 1) % 10 == 0:
        accuracy = 100 * correct / total
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {accuracy:.2f}%")

Epoch 10 | Loss: 0.0036 | Accuracy: 100.00%
Epoch 20 | Loss: 0.0004 | Accuracy: 100.00%
Epoch 30 | Loss: 0.0004 | Accuracy: 100.00%
Epoch 40 | Loss: 0.0002 | Accuracy: 100.00%
Epoch 50 | Loss: 0.0002 | Accuracy: 100.00%
Epoch 60 | Loss: 0.0002 | Accuracy: 100.00%
Epoch 70 | Loss: 0.0001 | Accuracy: 100.00%
Epoch 80 | Loss: 0.0002 | Accuracy: 100.00%
Epoch 90 | Loss: 0.0001 | Accuracy: 100.00%
Epoch 100 | Loss: 0.0001 | Accuracy: 100.00%


In [10]:
model.eval()

def predict(sentence):

    encoded = encode(sentence)
    encoded += [vocab["<PAD>"]] * (max_len - len(encoded))

    x = torch.tensor([encoded])

    with torch.no_grad():
        output = model(x)
        probability = torch.softmax(output, dim=1)
        prediction = probability.argmax(dim=1).item()

    print(f"Sentence : {sentence}")
    print(f"Predicted Class : {prediction}")
    print(f"Probability : {probability.squeeze().tolist()}")
    print()
    

predict("i love python")

Sentence : i love python
Predicted Class : 1
Probability : [6.98822332196869e-05, 0.9999301433563232]



In [11]:
torch.save(model.state_dict(), "../../models/lstm_sentiment.pth")